## Module 3: Architecture of Fastforward Neural Network

In [7]:
# Activation Functions
import torch
import torch.nn as nn

# 1. Sigmoid
sigmoid = nn.Sigmoid()
z = torch.tensor([-3.0, -1.0, 0.0, 1.0, 3.0])
print(f"Sigmoid = {sigmoid(z)}")

# 2. Tanh
tanh = nn.Tanh()
print(f"Tanh = {tanh(z)}")

# 3. ReLU
relu = nn.ReLU()
print(f"ReLU = {relu(z)}")

# 4. LeakyReLU
leaky_relu = nn.LeakyReLU(negative_slope = 0.01)
print(f"Leaky ReLU = {leaky_relu(z)}")

# 5. Softmax
softmax = nn.Softmax(dim = 1)
logits = torch.tensor([[2.0, 1.0, 0.5]])
print(f"Softmax = {softmax(logits)}")
print(f"Sum of all Probabilities = {softmax(logits).sum()}")

Sigmoid = tensor([0.0474, 0.2689, 0.5000, 0.7311, 0.9526])
Tanh = tensor([-0.9951, -0.7616,  0.0000,  0.7616,  0.9951])
ReLU = tensor([0., 0., 0., 1., 3.])
Leaky ReLU = tensor([-0.0300, -0.0100,  0.0000,  1.0000,  3.0000])
Softmax = tensor([[0.6285, 0.2312, 0.1402]])
Sum of all Probabilities = 0.9999999403953552


In [10]:
# Output Layer Designs
hidden_size = 0

# 1. Regression
nn.Linear(hidden_size, 1)

# 2. Binary Classification
nn.Linear(hidden_size, 1)
nn.Sigmoid()

# 3. MultiClass Classification
nn.Linear(hidden_size, 10)
nn.Softmax(dim = 1)

C:\Users\bisht\jupyter_env\tensorflow\torch_env\Lib\site-packages\torch\nn\modules\linear.py:124: UserWarning: Initializing zero-element tensors is a no-op
  init.kaiming_uniform_(self.weight, a=math.sqrt(5))


Softmax(dim=1)

In [13]:
# Structure of nn.Module Class
class MyNetwork(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return x

# nn.Sequential
model = nn.Sequential(
    nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 4), nn.ReLU(), nn.Linear(4, 1), nn.Sigmoid()
)

In [19]:
# Regression Network:
"""
Architecture:
      Input(in_features) → Linear(128) → ReLU
                         → Linear(64)  → ReLU
                         → Linear(32)  → ReLU
                         → Linear(1)   [no activation]
"""
class RegressionNet(nn.Module):
    def __init__(self, in_features):
        super().__init__()
        self.layer1 = nn.Linear(in_features, 128)
        self.layer2 = nn.Linear(128, 64)
        self.layer3 = nn.Linear(64, 32)
        self.output = nn.Linear(32, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.output(self.relu(self.layer3(self.relu(self.layer2(self.relu(self.layer1(x)))))))
        return x

model = RegressionNet(in_features = 10)
x_sample = torch.randn(16, 10)
output = model(x_sample)
print(f'Output Shape: {output.shape}')
print(f'Sample Output: {output[:3]}')

Output Shape: torch.Size([16, 1])
Sample Output: tensor([[-0.2012],
        [-0.1922],
        [-0.1718]], grad_fn=<SliceBackward0>)


In [20]:
# Binary Classification Network:
"""
Architecture:
      Input(in_features) → Linear(64) → ReLU
                         → Linear(32) → ReLU
                         → Linear(1)  → Sigmoid
"""
class BinaryClassificationNet(nn.Module):
    def __init__(self, in_features):
        super().__init__()
        self.layer1 = nn.Linear(in_features, 64)
        self.layer2 = nn.Linear(64, 32)
        self.layer3 = nn.Linear(32, 1)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.sigmoid(self.layer3(self.relu(self.layer2(self.relu(self.layer1(x))))))
        return x

model = BinaryClassificationNet(in_features = 8)
x_sample = torch.randn(32, 8)
output = model(x_sample)
print(f'Output Shape: {output.shape}')
print(f'Sample Output: {output[:3]}')
print(f'Predictions: {(output > 0.5).int()[:5].T}')

Output Shape: torch.Size([32, 1])
Sample Output: tensor([[0.4725],
        [0.4728],
        [0.5004]], grad_fn=<SliceBackward0>)
Predictions: tensor([[0, 0, 1, 0, 0]], dtype=torch.int32)


In [21]:
# MultiClass Classification Network:
"""
Architecture:
      Input(in_features) → Linear(128) → ReLU
                         → Linear(64)  → ReLU
                         → Linear(num_classes)  [no activation — CrossEntropyLoss handles softmax]
"""
class MulticlassNet(nn.Module):
    def __init__(self, in_features, num_classes):
        super().__init__()
        self.layer1 = nn.Linear(in_features, 128)
        self.layer2 = nn.Linear(128, 64)
        self.output = nn.Linear(64, num_classes)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.output(self.relu(self.layer2(self.relu(self.layer1(x)))))
        return x

model = MulticlassNet(in_features = 20, num_classes = 10)
x_sample = torch.randn(64, 20)
logits = model(x_sample)
print(f'Output Shape: {logits.shape}')

probs = torch.softmax(logits, dim = 1)
print(f'Prob Shape: {probs.shape}')
print(f'Row Sum: {probs[0].sum().item()}')

predicted_class = torch.argmax(probs, dim = 1)
print(f'Predicted Class: {predicted_class[:5]}')

Output Shape: torch.Size([64, 10])
Prob Shape: torch.Size([64, 10])
Row Sum: 0.9999999403953552
Predicted Class: tensor([3, 6, 6, 6, 6])


In [28]:
# Inspecting the model
model = MulticlassNet(in_features = 20, num_classes = 10)
print(f"Model Architecture:\n{model}")

total_params = sum(p.numel() for p in model.parameters())
print(f'Total Parameters: {total_params:,}')

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable Parameters: {trainable:,}')

print(f'Layer 1 weight shape: {model.layer1.weight.shape}')
print(f'Layer 1 bias shape: {model.layer1.bias.shape}')

Model Architecture:
MulticlassNet(
  (layer1): Linear(in_features=20, out_features=128, bias=True)
  (layer2): Linear(in_features=128, out_features=64, bias=True)
  (output): Linear(in_features=64, out_features=10, bias=True)
  (relu): ReLU()
)
Total Parameters: 11,594
Trainable Parameters: 11,594
Layer 1 weight shape: torch.Size([128, 20])
Layer 1 bias shape: torch.Size([128])


In [32]:
# torchinfo for detailed summary
# pip install torchinfo
from torchinfo import summary
summary(model, input_size = (64, 20))

Layer (type:depth-idx)                   Output Shape              Param #
MulticlassNet                            [64, 10]                  --
├─Linear: 1-1                            [64, 128]                 2,688
├─ReLU: 1-2                              [64, 128]                 --
├─Linear: 1-3                            [64, 64]                  8,256
├─ReLU: 1-4                              [64, 64]                  --
├─Linear: 1-5                            [64, 10]                  650
Total params: 11,594
Trainable params: 11,594
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.74
Input size (MB): 0.01
Forward/backward pass size (MB): 0.10
Params size (MB): 0.05
Estimated Total Size (MB): 0.15

In [33]:
# Putting All Together:
"""
  Args:
        in_features   (int):  Number of input features.
        hidden_sizes  (list): List of hidden layer sizes.
                              e.g., [128, 64, 32]
        out_features  (int):  Number of output neurons.
        task          (str):  'regression', 'binary', or 'multiclass'.
        dropout_rate  (float): Dropout probability (0 = no dropout).
"""
import torch
import torch.nn as nn

class FeedForwardNet(nn.Module):
    def __init__(self, in_features, hidden_sizes, out_features, task = 'multiclass', dropout_rate = 0.0):
        super().__init__()
        self.task = task
        layers = []
        prev_size = in_features

        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(prev_size, hidden_size))
            layers.append(nn.ReLU())

            if dropout_rate > 0:
                layers.append(nn.Dropout(p = dropout_rate))

            prev_size = hidden_size

        layers.append(nn.Linear(prev_size, out_features))

        if task == 'binary':
            layers.append(nn.Sigmoid())
        elif task == 'multiclass':
            pass
        elif task == 'regression':
            pass
        else:
            raise ValueError('You have entered unknown task value')

        self.network = nn.Sequential(*layers)

    def forward(self, x):
        x = self.network(x)
        return x

# Regression Model
reg_model = FeedForwardNet(in_features = 10, hidden_sizes = [128, 64, 32], out_features = 1, 
                           task = 'regression', dropout_rate = 0.0)
# Binary Model
bin_model = FeedForwardNet(in_features = 15, hidden_sizes = [64, 32], out_features = 1, 
                           task = 'binary', dropout_rate = 0.0)
# Multiclass Classification
mc_model = FeedForwardNet(in_features = 20, hidden_sizes = [256, 128, 64], out_features = 10, 
                           task = 'multiclass', dropout_rate = 0.0)

x = torch.randn(32, 20)
out = mc_model(x)
print(f'Multiclass Output Shape: {out.shape}')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
mc_model = mc_model.to(device)
x = x.to(device)
out = mc_model(x)
print(f'Output on {device}: {out.shape}')

total = sum(p.numel() for p in model.parameters())
print(f'Total Parameters: {total:,}')

Multiclass Output Shape: torch.Size([32, 10])
Output on cuda: torch.Size([32, 10])
Total Parameters: 11,594


In [40]:
# Mini Exercise: Building and inspecting three complete models -- one for each task

# 1. Regression Model
class HousePriceNet(nn.Module):
    """
    Predicts house price from 13 features (Boston Housing style).
    Architecture: 13 → 64 → 32 → 1
    """
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(13, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.fc3(self.relu(self.fc2(self.relu(self.fc1(x)))))
        return x

# 2. Binary Classification Model
class ChurnNet(nn.Module):
    """
    Predicts if a customer will churn (0 or 1) from 10 features.
    Architecture: 10 → 32 → 16 → 1 → Sigmoid
    """
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(10, 32)
        self.fc2 = nn.Linear(32, 16)
        self.fc3 = nn.Linear(16, 1)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.sigmoid(self.fc3(self.relu(self.fc2(self.relu(self.fc1(x))))))
        return x

# 3. Multiclass Classification Model
class IrisNet(nn.Module):
    """
    Classifies iris flowers into 3 species from 4 features.
    Architecture: 4 → 16 → 8 → 3
    """
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(4, 16)
        self.fc2 = nn.Linear(16, 8)
        self.fc3 = nn.Linear(8, 3)
        self.relu = nn.ReLU()
        
    def forward(self, x):
        x = self.fc3(self.relu(self.fc2(self.relu(self.fc1(x)))))
        return x

house_model = HousePriceNet()
churn_model = ChurnNet()
iris_model = IrisNet()

house_input = torch.randn(16, 13)
churn_input = torch.randn(16, 10)
iris_input = torch.randn(16, 4)

house_out = house_model(house_input)
churn_out = churn_model(churn_input)
iris_out = iris_model(iris_input)

print(f'House Price Output Shape: {house_out.shape}')
print(f'Churn Output Shape: {churn_out.shape}')
print(f'Iris Output Shape: {iris_out.shape}')

for name, model in [('HousePriceNet', house_model), ('ChurnNet', churn_model), ('IrisNet', iris_model)]:
    total = sum(p.numel() for p in model.parameters())
    print(f'{name} : {total:,} parameters')

print(f'\nHouse Price Predictions (first 3): {house_out[:3].detach()}')
print(f'Churn Probabilites (first 3): {churn_out[:3].detach()}')
print(f'Churn Predictions): {(churn_out > 0.5).int()[:3].T}')

iris_probs = torch.softmax(iris_out, dim = 1)
iris_pred = torch.argmax(iris_probs, dim = 1)
print(f'Iris Predicted Classes (first 5): {iris_pred[:5]}')
print(f'Iris Class Probabilities (first 3): {iris_probs[:3].detach()}')

House Price Output Shape: torch.Size([16, 1])
Churn Output Shape: torch.Size([16, 1])
Iris Output Shape: torch.Size([16, 3])
HousePriceNet : 3,009 parameters
ChurnNet : 897 parameters
IrisNet : 243 parameters

House Price Predictions (first 3): tensor([[-0.1108],
        [ 0.1636],
        [-0.0207]])
Churn Probabilites (first 3): tensor([[0.5087],
        [0.5010],
        [0.4982]])
Churn Predictions): tensor([[1, 1, 0]], dtype=torch.int32)
Iris Predicted Classes (first 5): tensor([1, 1, 1, 1, 1])
Iris Class Probabilities (first 3): tensor([[0.2423, 0.4247, 0.3329],
        [0.2491, 0.4194, 0.3315],
        [0.2740, 0.3957, 0.3304]])
